In [ ]:
# Import all packages
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

In [ ]:
# Data
df_all_stats = pd.read_csv("/content/stats (1).csv")

# Data
This Data is from the website Baseball Savant. https://baseballsavant.mlb.com/. Baseball Savant allows you to filter all prior Major League baseball season statistics and search by any parameters.

In [ ]:
# Sort the dataset by batters faced
df = df_all_stats.sort_values(by=['pa'], ascending=False)
df.head()

,"last_name, first_name",player_id,year,pa,k_percent,bb_percent,xwoba,exit_velocity_avg,whiff_percent,n_ff_formatted,...,n_ch_formatted,n_cu_formatted,n_si_formatted,n_fc_formatted,n_fs_formatted,n_kn_formatted,n_st_formatted,n_sv_formatted,n_fo_formatted,n_sc_formatted
110,"Webb, Logan",657277,2025,856,26.2,5.4,0.295,89.7,24.7,7.9,...,24.1,NaN,33.6,7.8,NaN,NaN,26.6,NaN,NaN,NaN
87,"Crochet, Garrett",676979,2025,814,31.3,5.7,0.266,87.7,29.4,35.9,...,4.3,NaN,16.0,27.7,0.0,NaN,16.0,NaN,NaN,NaN
11,"Gallen, Zac",668678,2025,813,21.5,8.1,0.320,90.1,23.7,45.0,...,16.0,23.5,2.5,0.1,NaN,NaN,NaN,NaN,NaN,NaN
25,"Sánchez, Cristopher",650911,2025,807,26.3,5.5,0.272,89.0,30.4,NaN,...,37.4,NaN,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,"Valdez, Framber",664285,2025,802,23.3,8.5,0.303,90.8,26.6,0.2,...,17.9,33.1,45.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Initial Analysis/Cleaning:

The main cleaning that was done here was the grouping of pitchers based on pitch arsenals. To do so I had to create rules for different pitch thresholds so that when I ran the python code it put them into the correct groups.

In [ ]:
# Find the mean, median and outliers for kk%-bb%, xWOBA.
# Calculate k_percent - bb_percent
df['k_minus_bb_percent'] = df['k_percent'] - df['bb_percent']

# Columns for analysis
analysis_cols = ['k_minus_bb_percent', 'xwoba']

# For each column, find mean, median, and outliers
results = {}
for col in analysis_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][['last_name, first_name', col]]

    results[col] = {
        'mean': mean_val,
        'median': median_val,
        'outliers': outliers.to_dict(orient='records')
    }

print("Analysis Results:")
for col, data in results.items():
    print(f"\n--- {col} ---")
    print(f"Mean: {data['mean']:.2f}")
    print(f"Median: {data['median']:.2f}")
    print(f"Outliers ({len(data['outliers'])} found):")
    if data['outliers']:
        for outlier in data['outliers']:
            print(f"  {outlier['last_name, first_name']}: {outlier[col]:.2f}")

Analysis Results:

--- k_minus_bb_percent ---
Mean: 14.17
Median: 13.50
Outliers (3 found):
  Skubal, Tarik: 27.80
  Wheeler, Zack: 27.70
  Gilbert, Logan: 26.50

--- xwoba ---
Mean: 0.32
Median: 0.32
Outliers (2 found):
  Senzatela, Antonio: 0.40
  Wheeler, Zack: 0.25


In [ ]:
# Rename columns
renamed_cols = {}
for col in df.columns:
    if col.startswith('n_') and col.endswith('_formatted'):
        pitch_type = col[2:-10] # Extract the pitch type (e.g., 'ff' from 'n_ff_formatted')
        renamed_cols[col] = f'{pitch_type}_percentage'

df = df.rename(columns=renamed_cols)
display(df.head())

,"last_name, first_name",player_id,year,pa,k_percent,bb_percent,xwoba,exit_velocity_avg,whiff_percent,ff_percentage,...,cu_percentage,si_percentage,fc_percentage,fs_percentage,kn_percentage,st_percentage,sv_percentage,fo_percentage,sc_percentage,k_minus_bb_percent
110,"Webb, Logan",657277,2025,856,26.2,5.4,0.295,89.7,24.7,7.9,...,NaN,33.6,7.8,NaN,NaN,26.6,NaN,NaN,NaN,20.8
87,"Crochet, Garrett",676979,2025,814,31.3,5.7,0.266,87.7,29.4,35.9,...,NaN,16.0,27.7,0.0,NaN,16.0,NaN,NaN,NaN,25.6
11,"Gallen, Zac",668678,2025,813,21.5,8.1,0.320,90.1,23.7,45.0,...,23.5,2.5,0.1,NaN,NaN,NaN,NaN,NaN,NaN,13.4
25,"Sánchez, Cristopher",650911,2025,807,26.3,5.5,0.272,89.0,30.4,NaN,...,NaN,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.8
18,"Valdez, Framber",664285,2025,802,23.3,8.5,0.303,90.8,26.6,0.2,...,33.1,45.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.8


## Grouping based on pitch arsenals

In [ ]:
required_pitch_cols = [
    'ff_percentage', 'si_percentage', 'fc_percentage', 'sl_percentage',
    'sw_percentage', 'cu_percentage', 'ch_percentage', 'kn_percentage',
    'st_percentage', 'sv_percentage', 'fo_percentage', 'sc_percentage', 'fs_percentage'
]

# Filter required_pitch_cols to only include those present in the DataFrame
actual_pitch_cols = [col for col in required_pitch_cols if col in df.columns]

# Add a column for the pitchers throwing hand
if 'pitcher_hand' not in df.columns:
    df['pitcher_hand'] = 'R' # Default to Right-handed for now

# Define pitch usage thresholds
HIGH_PRIMARY_THRESHOLD = 25.0
HIGH_SECONDARY_THRESHOLD = 10.0
LOW_FF_THRESHOLD = 30.0
CH_SPECIALIST_THRESHOLD = 20.0
BREAKING_BALL_SECONDARY_THRESHOLD = 15.0
SINKER_FIRST_PRIMARY_THRESHOLD = 20.0
MIN_DEEP_MIX_USAGE = 5.0
MIN_DEEP_MIX_PITCH_COUNT = 4

def assign_pitcher_cluster(row):
    # Assigning pitchers to a cluster based on pitch usage thresholds
    pitch_data = {col: row[col] if pd.notna(row[col]) else 0.0 for col in actual_pitch_cols}
    ff_p = pitch_data.get('ff_percentage', 0.0)
    si_p = pitch_data.get('si_percentage', 0.0)
    fc_p = pitch_data.get('fc_percentage', 0.0)
    sl_p = pitch_data.get('sl_percentage', 0.0)
    sw_p = pitch_data.get('sw_percentage', 0.0)
    cu_p = pitch_data.get('cu_percentage', 0.0)
    ch_p = pitch_data.get('ch_percentage', 0.0)
    kn_p = pitch_data.get('kn_percentage', 0.0)

    pitcher_hand = row.get('pitcher_hand', 'Unknown')

    # Determine primary pitch and its percentage for 'Deep Mix' and 'Sinker First'
    pitch_percentages_series = pd.Series({
        'ff': ff_p, 'si': si_p, 'fc': fc_p, 'sl': sl_p,
        'sw': sw_p, 'cu': cu_p, 'ch': ch_p, 'kn': kn_p
    })
    significant_pitches = pitch_percentages_series[pitch_percentages_series >= MIN_DEEP_MIX_USAGE] if (pitch_percentages_series[pitch_percentages_series >= MIN_DEEP_MIX_USAGE]).any() else pd.Series([])
    max_pitch_percentage = significant_pitches.max() if not significant_pitches.empty else 0.0
    max_pitch_type = significant_pitches.idxmax() if not significant_pitches.empty else None

    # Applied pitching groups
    # I had to do this in a specific order so that all groups would be represented, otherwise majority of pitchers would have ended upn in "Deep Mix"

    # 1. Lefty Changeup Heavy
    if pitcher_hand == 'L' and ch_p > CH_SPECIALIST_THRESHOLD:
        return 9, 'Lefty Changeup Specialist'

    # 2. Changeup Specialist
    if pitcher_hand == 'R' and ch_p > CH_SPECIALIST_THRESHOLD:
        return 8, 'Changeup Specialist'

    # 3. Four seam / sweeper
    if ff_p >= HIGH_PRIMARY_THRESHOLD and sw_p >= HIGH_SECONDARY_THRESHOLD:
        return 1, 'Four Seam + Sweeper'

    # 4. Four seam / 12/6
    if ff_p >= HIGH_PRIMARY_THRESHOLD and cu_p >= HIGH_SECONDARY_THRESHOLD:
        return 2, '4 seam + 12/6'

    # 5. Four Seam and changeup/Splitter
    if pitcher_hand == 'R' and ff_p >= HIGH_PRIMARY_THRESHOLD and ch_p >= HIGH_SECONDARY_THRESHOLD:
        return 3, 'Four Seam and Changeup/Splitter'

    # 6. Cutter/Sinker
    if fc_p >= HIGH_PRIMARY_THRESHOLD and si_p >= HIGH_SECONDARY_THRESHOLD:
        return 4, 'Cutter/Sinker'

    # 7. Sinker primary
    # "high si_percentage as primary pitch + breaking ball secondary"
    if max_pitch_type == 'si' and si_p >= SINKER_FIRST_PRIMARY_THRESHOLD and (sl_p + cu_p) > BREAKING_BALL_SECONDARY_THRESHOLD:
        return 6, 'Primary Sinker'

    # 8. Big Arsenal
    # No single pitch above HIGH_PRIMARY_THRESHOLD (now 25%) AND 4+ pitch arsenal above MIN_DEEP_MIX_USAGE (5%)
    if max_pitch_percentage < HIGH_PRIMARY_THRESHOLD and len(significant_pitches) >= MIN_DEEP_MIX_PITCH_COUNT:
        return 7, 'Big Arsenal'

    # Default assignment to Deep Mix if no other rule matches (ensures every pitcher is assigned)
    return 7, 'Big Arsenal'

# Apply the function to create new columns for cluster group and name
df[['cluster_group', 'cluster_name']] = df.apply(assign_pitcher_cluster, axis=1, result_type='expand')

print("--- Cluster Analysis Results ---")


print("\nGroup Sizes:")
group_sizes = df['cluster_name'].value_counts().sort_index()
print(group_sizes)

display(df.head())

--- Cluster Analysis Results ---

Group Sizes:
cluster_name
4 seam + 12/6                      42
Big Arsenal                        31
Changeup Specialist                23
Cutter/Sinker                       8
Four Seam and Changeup/Splitter    12
Primary Sinker                     11
Name: count, dtype: int64


,"last_name, first_name",player_id,year,pa,k_percent,bb_percent,xwoba,exit_velocity_avg,whiff_percent,ff_percentage,...,fs_percentage,kn_percentage,st_percentage,sv_percentage,fo_percentage,sc_percentage,k_minus_bb_percent,pitcher_hand,cluster_group,cluster_name
110,"Webb, Logan",657277,2025,856,26.2,5.4,0.295,89.7,24.7,7.9,...,NaN,NaN,26.6,NaN,NaN,NaN,20.8,R,8,Changeup Specialist
87,"Crochet, Garrett",676979,2025,814,31.3,5.7,0.266,87.7,29.4,35.9,...,0.0,NaN,16.0,NaN,NaN,NaN,25.6,R,4,Cutter/Sinker
11,"Gallen, Zac",668678,2025,813,21.5,8.1,0.320,90.1,23.7,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,13.4,R,2,4 seam + 12/6
25,"Sánchez, Cristopher",650911,2025,807,26.3,5.5,0.272,89.0,30.4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.8,R,8,Changeup Specialist
18,"Valdez, Framber",664285,2025,802,23.3,8.5,0.303,90.8,26.6,0.2,...,NaN,NaN,NaN,NaN,NaN,NaN,14.8,R,6,Primary Sinker


# Research Questions

# Q1: Does Pitch arsenal effect K-BB%?

In [ ]:
# K-BB rate analysis for mean, median, standard deviation
kbb_analysis = df.groupby('cluster_name')['k_minus_bb_percent'].agg(['mean', 'median', 'std', 'count']).sort_values(by='mean', ascending=False)
display(kbb_analysis)

graph = px.bar(
    kbb_analysis.reset_index(),
    x='cluster_name',
    y='mean',
    error_y='std',
    title='Mean K-BB% by Pitch Arsenal Group (with Standard Deviation)',
    labels={'cluster_name': 'Pitch Arsenal Group', 'mean': 'Mean K-BB%'},
    template='plotly_white'
)
graph.update_layout(xaxis_title_text='Pitch Arsenal Group', yaxis_title_text='Mean K-BB%')
graph.show()

,mean,median,std,count
cluster_name,,,,
Four Seam and Changeup/Splitter,15.308333,14.05,4.764921,12
Big Arsenal,15.229032,14.40,6.494931,31
Changeup Specialist,14.495652,14.20,4.998498,23
Cutter/Sinker,13.850000,14.00,6.848149,8
4 seam + 12/6,13.300000,13.45,4.415659,42
Primary Sinker,12.772727,11.90,2.432320,11


# Q2: Does Pitch arsenal effect xWOBA?

In [ ]:
# xWOBA analysis for mean, median, standard deviation
xwoba_analysis = df.groupby('cluster_name')['xwoba'].agg(['mean', 'median', 'std', 'count']).sort_values(by='mean', ascending=False)
display(xwoba_analysis)

graph = px.bar(
    xwoba_analysis.reset_index(),
    x='cluster_name',
    y='mean',
    error_y='std',
    title='Mean xWOBA by Pitch Arsenal Group (with Standard Deviation)',
    labels={'cluster_name': 'Pitch Arsenal Group', 'mean': 'Mean xWOBA'},
    template='plotly_white'
)
graph.update_layout(xaxis_title_text='Pitch Arsenal Group', yaxis_title_text='Mean xWOBA')
graph.show()

,mean,median,std,count
cluster_name,,,,
4 seam + 12/6,0.324286,0.321,0.026403,42
Primary Sinker,0.317273,0.316,0.022499,11
Big Arsenal,0.315097,0.322,0.031101,31
Changeup Specialist,0.314000,0.316,0.026973,23
Cutter/Sinker,0.310625,0.310,0.029169,8
Four Seam and Changeup/Splitter,0.304417,0.309,0.023365,12


# Q3: Does Pitch arsenal effect Average Exit Velocity

In [ ]:
# Average Exit Velocity analysis for mean, median, standard deviation
exit_velocity_analysis = df.groupby('cluster_name')['exit_velocity_avg'].agg(['mean', 'median', 'std', 'count']).sort_values(by='mean', ascending=False)
display(exit_velocity_analysis)

graph = px.bar(
    exit_velocity_analysis.reset_index(),
    x='cluster_name',
    y='mean',
    error_y='std',
    title='Mean Average Exit Velocity by Pitch Arsenal Group (with Standard Deviation)',
    labels={'cluster_name': 'Pitch Arsenal Group', 'mean': 'Mean Average Exit Velocity'},
    template='plotly_white'
)
graph.update_layout(xaxis_title_text='Pitch Arsenal Group', yaxis_title_text='Mean Average Exit Velocity')
graph.show()

,mean,median,std,count
cluster_name,,,,
4 seam + 12/6,90.007143,90.3,1.356716,42
Big Arsenal,89.680645,89.8,1.118457,31
Primary Sinker,89.572727,90.2,1.494718,11
Four Seam and Changeup/Splitter,89.266667,88.9,1.056007,12
Cutter/Sinker,89.050000,89.3,1.178377,8
Changeup Specialist,89.004348,89.0,1.386125,23


# Q4: Does Pitch Arsenal Effect Whiff%?

In [ ]:
# Whiff Rate analysis for mean, median, standard deviation
whiff_analysis = df.groupby('cluster_name')['whiff_percent'].agg(['mean', 'median', 'std', 'count']).sort_values(by='mean', ascending=False)
display(whiff_analysis)

graph = px.bar(
    whiff_analysis.reset_index(),
    x='cluster_name',
    y='mean',
    error_y='std',
    title='Mean Whiff Rate by Pitch Arsenal Group (with Standard Deviation)',
    labels={'cluster_name': 'Pitch Arsenal Group', 'mean': 'Mean Whiff Rate'},
    template='plotly_white'
)
graph.update_layout(xaxis_title_text='Pitch Arsenal Group', yaxis_title_text='Mean Whiff Rate')
graph.show()

,mean,median,std,count
cluster_name,,,,
Four Seam and Changeup/Splitter,25.091667,24.45,4.424202,12
Changeup Specialist,24.600000,24.10,3.848494,23
Big Arsenal,24.354839,24.00,4.655594,31
4 seam + 12/6,23.738095,23.85,3.673435,42
Primary Sinker,23.418182,24.00,2.625954,11
Cutter/Sinker,23.362500,22.85,4.064810,8


# Export the cleaned/grouped data to be used for visualizations

In [ ]:
output_filename = 'pitch_arsenal_data_with_groupings.csv'
df.to_csv(output_filename, index=False)
